In [20]:
import pandas as pd
import numpy as np
import sklearn as skl
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
tabla = pd.read_csv('siniestros_viales_victimas.csv', sep=';')
# 1. Guardas el enlace "Raw" en una variable (reemplaza esto con tu URL real)
url_github = 'https://raw.githubusercontent.com/Marian2057/The_Outliers/refs/heads/main/Proyecto/Data/siniestros_viales_victimas.csv'

# 2. Pandas lee la URL directamente de la misma forma que un archivo local
tabla = pd.read_csv(url_github, sep=';')



C:\Users\pim\AppData\Local\Temp\ipykernel_740\992307425.py:1: DtypeWarning: Columns (0,1,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  tabla = pd.read_csv('siniestros_viales_victimas.csv', sep=';')
C:\Users\pim\AppData\Local\Temp\ipykernel_740\992307425.py:6: DtypeWarning: Columns (0,1,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  tabla = pd.read_csv(url_github, sep=';')


In [22]:
tabla.head(5)

,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
0,LC-2019-0000647,2019-01-01,2019.0,MOTO,M,54,GRAVE,SD,NaN
1,LC-2019-0000600,2019-01-01,2019.0,SD,F,1,LEVE,SD,NaN
2,LC-2019-0000136,2019-01-01,2019.0,SD,F,21,LEVE,SD,NaN
3,LC-2019-0000082,2019-01-01,2019.0,SD,F,32,LEVE,SD,NaN
4,LC-2019-0000194,2019-01-01,2019.0,SD,F,33,LEVE,SD,NaN


In [23]:
# Ver la cantidad de filas, columnas y los tipos de datos actuales
tabla.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_siniestro                 62076 non-null  object 
 1   fecha_siniestro              62076 non-null  object 
 2   anio_siniestro               62076 non-null  float64
 3   modo_desplazamiento_victima  62076 non-null  object 
 4   sexo_victima                 62076 non-null  object 
 5   edad_victima                 62076 non-null  object 
 6   GRAVEdad_victima             62076 non-null  object 
 7   rol_victima                  61865 non-null  object 
 8   fecha_fallecimiento_victima  610 non-null    object 
dtypes: float64(1), object(8)
memory usage: 72.0+ MB


In [24]:
# Ver un resumen estadístico de las columnas numéricas
tabla.describe()

,anio_siniestro
count,62076.000000
mean,2021.636784
std,1.779534
min,2019.000000
25%,2020.000000
50%,2022.000000
75%,2023.000000
max,2024.000000


In [25]:
# Contar cuántos valores nulos hay por columna
tabla.isnull().sum()

id_siniestro                    986499
fecha_siniestro                 986499
anio_siniestro                  986499
modo_desplazamiento_victima     986499
sexo_victima                    986499
edad_victima                    986499
GRAVEdad_victima                986499
rol_victima                     986710
fecha_fallecimiento_victima    1047965
dtype: int64

1. Eliminar las filas vacías
- No tiene sentido procesar un millón de filas vacías. Lo más seguro es eliminar cualquier fila que no tenga un identificador de siniestro.

In [26]:
# Eliminar filas donde 'id_siniestro' sea nulo
tabla.dropna(subset=['id_siniestro'], inplace=True)

# Verificar cuántas filas quedaron
print("Total de filas tras la limpieza:", len(tabla))

Total de filas tras la limpieza: 62076


2. Tratar los valores "SD" (Sin Datos)
- Como vimos antes, es probable que la edad o las fechas tengan el texto "SD". Vamos a estandarizarlo a un nulo real (NaN) que Pandas pueda entender.

In [27]:
# Reemplazar "SD" por NaN (nulo real de numpy/pandas) en todo el DataFrame
tabla.replace('SD', np.nan, inplace=True)

# Ahora sí, si vuelves a hacer tabla.isnull().sum(), verás los nulos reales

3. Corregir los Tipos de Datos (Dtypes)
- Darle a cada columna el formato correcto. Esto es crucial si luego vas a conectar estos datos a herramientas de visualización para armar tus tableros.

In [28]:
# 1. Convertir fechas a formato datetime
tabla['fecha_siniestro'] = pd.to_datetime(tabla['fecha_siniestro'], errors='coerce')
tabla['fecha_fallecimiento_victima'] = pd.to_datetime(tabla['fecha_fallecimiento_victima'], errors='coerce')

# 2. Convertir edad a número (estaba como texto 'object')
tabla['edad_victima'] = pd.to_numeric(tabla['edad_victima'], errors='coerce')
# Convertir la columna a formato entero que acepta nulos
tabla['edad_victima'] = tabla['edad_victima'].astype('Int64')

# 3. Convertir el año a número entero 
# (estaba como float64 porque Pandas usa decimales cuando hay filas nulas)
tabla['anio_siniestro'] = tabla['anio_siniestro'].astype(int)

In [29]:
tabla.head(5)

,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
0,LC-2019-0000647,2019-01-01,2019,MOTO,M,54,GRAVE,NaN,NaT
1,LC-2019-0000600,2019-01-01,2019,NaN,F,1,LEVE,NaN,NaT
2,LC-2019-0000136,2019-01-01,2019,NaN,F,21,LEVE,NaN,NaT
3,LC-2019-0000082,2019-01-01,2019,NaN,F,32,LEVE,NaN,NaT
4,LC-2019-0000194,2019-01-01,2019,NaN,F,33,LEVE,NaN,NaT
